In [ ]:
import pandas as pd
from transformers import RobertaForSequenceClassification, RobertaTokenizer
from datasets import Dataset
import torch
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

In [ ]:
df = pd.read_csv("2021_Patient_level_linkage_withIDs.csv", dtype="string")

In [ ]:
df['Match Status'] = df['Match Status'].replace({'Match': '1', 'Non-Match': '0'})
df['Match Status'] = df['Match Status'].astype(int)
df.rename(columns={'Match Status': 'labels'}, inplace=True)

In [ ]:

model_path = "roberta-linkage-classifier"
tokenizer = RobertaTokenizer.from_pretrained(model_path)
model = RobertaForSequenceClassification.from_pretrained(model_path)

In [ ]:

def serialize_record(record_dict):
    serialized_str = ""
    for col_name, value in record_dict.items():
        serialized_str += f" [COL] {col_name} [VAL] {value}"
    return serialized_str.strip()


In [ ]:
def create_input_pair(row):
    record1_dict = {
        "First Name": row["record1 First Name"],
        "Middle Name": row["record1 Middle Name"],
        "Last Name": row["record1 Last Name"],
        "Date of Birth": row["record1 Date of Birth"],
        "SSN": row["record1 SSN"],
        "Sex": row["record1 Sex"],
        "Address": row["record1 Address"],
    }
    record2_dict = {
        "First Name": row["record2 First Name"],
        "Middle Name": row["record2 Middle Name"],
        "Last Name": row["record2 Last Name"],
        "Date of Birth": row["record2 Date of Birth"],
        "SSN": row["record2 SSN"],
        "Sex": row["record2 Sex"],
        "Address": row["record2 Address"],
    }
    
    serialized_r1 = serialize_record(record1_dict)
    serialized_r2 = serialize_record(record2_dict)
    
    return serialized_r1, serialized_r2


In [ ]:
def tokenize_function(examples):

    record1_texts = []
    record2_texts = []
    
    for i in range(len(examples["labels"])):
        row = {
            "record1 First Name": examples["record1 First Name"][i],
            "record1 Middle Name": examples["record1 Middle Name"][i],
            "record1 Last Name": examples["record1 Last Name"][i],
            "record1 Date of Birth": examples["record1 Date of Birth"][i],
            "record1 SSN": examples["record1 SSN"][i],
            "record1 Sex": examples["record1 Sex"][i],
            "record1 Address": examples["record1 Address"][i],
            "record2 First Name": examples["record2 First Name"][i],
            "record2 Middle Name": examples["record2 Middle Name"][i],
            "record2 Last Name": examples["record2 Last Name"][i],
            "record2 Date of Birth": examples["record2 Date of Birth"][i],
            "record2 SSN": examples["record2 SSN"][i],
            "record2 Sex": examples["record2 Sex"][i],
            "record2 Address": examples["record2 Address"][i],
        }
        
        r1, r2 = create_input_pair(row)
        record1_texts.append(r1)
        record2_texts.append(r2)
    

    tokenized_batch = tokenizer(
        record1_texts,
        record2_texts,
        padding="max_length",
        truncation=True,
        max_length=256,
    )
    
    tokenized_batch["labels"] = examples["labels"]
    return tokenized_batch


In [ ]:
dataset = Dataset.from_pandas(df)
tokenized_dataset = dataset.map(tokenize_function, batched=True, remove_columns=dataset.column_names)

In [ ]:

tokenized_dataset = tokenized_dataset.with_format("torch")

In [ ]:
from tqdm import tqdm
import torch
from sklearn.metrics import classification_report, accuracy_score
import torch.nn.functional as F



model = model.to("cuda")

model.eval()


batch_size = 1000
dataloader = torch.utils.data.DataLoader(tokenized_dataset, batch_size=batch_size)


predictions = []
confidences = []
labels = []


with torch.no_grad():
    for batch in tqdm(dataloader, desc="Evaluating", unit="batch"):

        inputs = {k: v.to("cuda") for k, v in batch.items() if k != "labels"}
        batch_labels = batch["labels"].to("cuda")

        

        outputs = model(**inputs)
        preds = torch.argmax(outputs.logits, dim=-1)
        probs = F.softmax(outputs.logits, dim=-1)
        confidence_scores = probs[torch.arange(probs.size(0)), preds]
        

        predictions.extend(preds.cpu().numpy())
        confidences.extend(confidence_scores.cpu().numpy())
        labels.extend(batch_labels.cpu().numpy())


predictions = np.array(predictions)
confidences = np.array(confidences)
labels = np.array(labels)


accuracy = accuracy_score(labels, predictions)
print("Accuracy:", accuracy)
print("\nClassification Report:\n", classification_report(labels, predictions, digits=6))


In [ ]:
df['prediction'] = predictions
df.to_csv("RoBERTa_Finetuned_Inference_Results.csv", index=False)